# 05A quick check: GAFF2 parameters in OpenMM (optional)

**Workflow:** Optional quick check of 05A parameters: 05A → 05A_quick_check_openmm

Optional: check that the 05A GAFF2 parameters load, minimise and run a few steps in OpenMM, without water. A fast test before building a solvated system. Not a physical result.

It reads the `prmtop` and `inpcrd` files written by `05A_amber_gaff2_parameters.ipynb` and writes only to `openmm/dry_polymer/`. It does not solvate the system and does not submit HPC jobs.

## Workflow Scope

- Engine: OpenMM
- System: one PHA oligomer, no water, usually no periodic box
- Input (written by 05A): `examples/output/md_tests/<SYSTEM>/gaff2/<SYSTEM>.prmtop` and `<SYSTEM>.inpcrd`
- Output: `examples/output/md_tests/<SYSTEM>/openmm/dry_polymer/`
- Stages: minimisation, short NVT, NPT only if the input already has box vectors, short production

Use the same system as in 05A (default `P3HB_4`). `RUN_OPENMM_DRY` is `False` by default, so Run All only checks paths.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd() if (Path.cwd() / "src" / "iphasimulator").exists() else Path.cwd().parent
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

output_root = repo_root / "examples" / "output"
from iphasimulator.naming import oligomer_name

system_name = oligomer_name("3HB", 4)
md_root = output_root / "md_tests" / system_name
print(f"Repository: {repo_root}")
print(f"System: {system_name}")
print(f"MD output root: {md_root}")

gaff2_dir = md_root / "gaff2"
openmm_dry_dir = md_root / "openmm" / "dry_polymer"
prmtop_path = gaff2_dir / f"{system_name}.prmtop"
inpcrd_path = gaff2_dir / f"{system_name}.inpcrd"

{
    "prmtop": prmtop_path,
    "prmtop_exists": prmtop_path.exists(),
    "inpcrd": inpcrd_path,
    "inpcrd_exists": inpcrd_path.exists(),
    "openmm_dry_dir": openmm_dry_dir,
}

## Check OpenMM

This only checks that the current Python environment can import OpenMM.


In [ ]:
from iphasimulator.simulation_openmm_amber_runner import openmm_available

openmm_available()

## Short Local Settings

These settings are intentionally small. Increase them only after the dry workflow runs cleanly.


In [ ]:
openmm_settings = {
    "minimization_max_iterations": 200,
    "nvt_steps": 100,
    "npt_steps": 100,
    "production_steps": 100,
    "report_interval": 10,
    "temperature_kelvin": 300.0,
    "pressure_bar": 1.0,
    "platform_name": None,  # set to "CUDA" for a local GPU check
    "platform_precision": "mixed",
}

openmm_settings

## Run Dry OpenMM

This cell is disabled by default so importing the notebook never creates trajectories or logs. Set `RUN_OPENMM_DRY = True` when the GAFF2 inputs exist and you are ready to write local output files.


In [ ]:
from iphasimulator.simulation_openmm_amber_runner import run_openmm_with_amber_topology

RUN_OPENMM_DRY = False

if RUN_OPENMM_DRY:
    openmm_dry_outputs = run_openmm_with_amber_topology(
        prmtop_path,
        inpcrd_path,
        openmm_dry_dir,
        **openmm_settings,
    )
    openmm_dry_outputs
else:
    print("Set RUN_OPENMM_DRY = True to run the dry OpenMM validation workflow.")

## Expected Dry Outputs

Generated trajectories, checkpoints, and logs live under `examples/output/`, which is ignored by Git by default.


In [ ]:
expected_outputs = [
    "minimized.pdb",
    "nvt.dcd",
    "nvt.log",
    "npt.log",
    "production.dcd",
    "production.log",
    "state.xml",
    "final.pdb",
    "openmm_summary.log",
]

{filename: (openmm_dry_dir / filename).exists() for filename in expected_outputs}